<a href="https://colab.research.google.com/github/Asofia29/QIIME2/blob/main/QIIME2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### QIIME2 HANDS-ON TUTORIAL

You’ll use QIIME 2 to perform an analysis of human microbiome samples from two individuals at four body sites at five timepoints, the first of which immediately followed antibiotic usage.
A study based on these samples was originally published in Caporaso et al. (2011).

The data used in this tutorial were sequenced on an Illumina HiSeq using the Earth Microbiome Project hypervariable region 4 (V4) 16S rRNA sequencing protocol.

### Downloading dataset and metadata files

In [ ]:
!wget \
  -O 'sample-metadata.tsv' \
  'https://docs.qiime2.org/2024.10/data/tutorials/moving-pictures-usage/sample-metadata.tsv'

--2024-12-14 19:09:59--  https://docs.qiime2.org/2024.10/data/tutorials/moving-pictures-usage/sample-metadata.tsv
Resolving docs.qiime2.org (docs.qiime2.org)... 172.67.186.144, 104.21.84.49, 2606:4700:3035::ac43:ba90, ...
Connecting to docs.qiime2.org (docs.qiime2.org)|172.67.186.144|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2096 (2.0K) [application/octet-stream]
Saving to: ‘sample-metadata.tsv’

sample-metadata.tsv 100%[===================>]   2.05K  --.-KB/s    in 0s      

2024-12-14 19:10:01 (6.77 MB/s) - ‘sample-metadata.tsv’ saved [2096/2096]



In [ ]:
!wget \
  -O 'emp-single-end-sequences.zip' \
  'https://docs.qiime2.org/2024.10/data/tutorials/moving-pictures-usage/emp-single-end-sequences.zip'

!unzip -d emp-single-end-sequences emp-single-end-sequences.zip


--2024-12-14 19:18:23--  https://docs.qiime2.org/2024.10/data/tutorials/moving-pictures-usage/emp-single-end-sequences.zip
Resolving docs.qiime2.org (docs.qiime2.org)... 172.67.186.144, 104.21.84.49, 2606:4700:3035::ac43:ba90, ...
Connecting to docs.qiime2.org (docs.qiime2.org)|172.67.186.144|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29096655 (28M) [application/zip]
Saving to: ‘emp-single-end-sequences.zip’

emp-single-end-sequ 100%[===================>]  27.75M  10.9MB/s    in 2.5s    

2024-12-14 19:18:27 (10.9 MB/s) - ‘emp-single-end-sequences.zip’ saved [29096655/29096655]

Archive:  emp-single-end-sequences.zip
  inflating: emp-single-end-sequences/sequences.fastq.gz  
  inflating: emp-single-end-sequences/barcodes.fastq.gz  


### Importing fastq files to create Artifacts

In [ ]:
!qiime tools import \
  --type 'EMPSingleEndSequences' \
  --input-path emp-single-end-sequences \
  --output-path emp-single-end-sequences.qza

Imported emp-single-end-sequences as EMPSingleEndDirFmt to emp-single-end-sequences.qza


### Demultiplexing sequences (Optional step)

Demultiplexing sequences is the process of mapping every sequencing read back to it's original sample using the Barcodes that are attached to it.
This is an optional step, since mostly the sequences are already provided in the demultiplexed format.


In [ ]:
!qiime demux emp-single \
  --i-seqs emp-single-end-sequences.qza \
  --m-barcodes-file sample-metadata.tsv \
  --m-barcodes-column barcode-sequence \
  --o-per-sample-sequences demux.qza \
  --o-error-correction-details demux-details.qza

Saved SampleData[SequencesWithQuality] to: demux.qza
Saved ErrorCorrectionDetails to: demux-details.qza


### Visualizing the demultiplexed sequences quality control graphs

In [ ]:
!qiime demux summarize \
  --i-data demux.qza \
  --o-visualization demux.qzv

Saved Visualization to: demux.qzv


### Denoising sequences : Performing sequence quality control

Denoising sequences refers to the process of filtering out reads that are of inferior quality, chimeric sequences and duplicates.


The dada2 denoise-single method requires two parameters that are used in quality filtering: --p-trim-left m, which trims off the first m bases of each sequence, and --p-trunc-len n which truncates each sequence at position n. This allows the user to remove low quality regions of the sequences.

To determine what values to pass for these two parameters, you should review the Interactive Quality Plot tab in the demux.qzv file that was generated by qiime demux summarize above.


In [ ]:
!qiime dada2 denoise-single \
  --i-demultiplexed-seqs demux.qza \
  --p-trim-left 0 \
  --p-trunc-len 120 \
  --o-representative-sequences rep-seqs-dada2.qza \
  --o-table table-dada2.qza \
  --o-denoising-stats stats-dada2.qza

Saved FeatureTable[Frequency] to: table-dada2.qza
Saved FeatureData[Sequence] to: rep-seqs-dada2.qza
Saved SampleData[DADA2Stats] to: stats-dada2.qza


## (Operational Taxonomic Units) OTU vs ASV (Amplicon Sequence Variant)

![OTU vs ASV](Operational_Taxonomic_Units.png)


# ![OTU vs ASV](Amplicon_Sequence_Variants.png)

In [ ]:
#Renaming the files
!mv rep-seqs-dada2.qza rep-seqs.qza
!mv table-dada2.qza table.qza

mv: cannot stat 'rep-seqs-dada2.qza': No such file or directory
mv: cannot stat 'table-dada2.qza': No such file or directory


### Generating feature table summary and visualizing representative sequences

After the quality filtering step completes, you’ll want to explore the resulting data. You can do this using the following two commands, which will create visual summaries of the data.

The feature-table summarize command will give you information on how many sequences are associated with each sample and with each feature, histograms of those distributions, and some related summary statistics.


The feature-table tabulate-seqs command will provide a mapping of feature IDs to sequences, and provide links to easily BLAST each sequence against the NCBI nt database

In [ ]:
!qiime feature-table summarize \
  --i-table table.qza \
  --o-visualization table.qzv \
  --m-sample-metadata-file sample-metadata.tsv
!qiime feature-table tabulate-seqs \
  --i-data rep-seqs.qza \
  --o-visualization rep-seqs.qzv

Saved Visualization to: table.qzv
Saved Visualization to: rep-seqs.qzv


### Phylogenetic analysis : Generation of rooted and unrooted trees

QIIME supports several phylogenetic diversity metrics, including Faith’s Phylogenetic Diversity and weighted and unweighted UniFrac. In addition to counts of features per sample (i.e., the data in the FeatureTable[Frequency] QIIME 2 artifact), these metrics require a rooted phylogenetic tree relating the features to one another. This information will be stored in a Phylogeny[Rooted] QIIME 2 artifact. To generate a phylogenetic tree we will use align-to-tree-mafft-fasttree pipeline from the q2-phylogeny plugin.

First, the pipeline uses the mafft program to perform a multiple sequence alignment of the sequences in our FeatureData[Sequence] to create a FeatureData[AlignedSequence] QIIME 2 artifact. Next, the pipeline masks (or filters) the alignment to remove positions that are highly variable. These positions are generally considered to add noise to a resulting phylogenetic tree. Following that, the pipeline applies FastTree to generate a phylogenetic tree from the masked alignment. The FastTree program creates an unrooted tree, so in the final step in this section midpoint rooting is applied to place the root of the tree at the midpoint of the longest tip-to-tip distance in the unrooted tree.

In [ ]:
!qiime phylogeny align-to-tree-mafft-fasttree \
  --i-sequences rep-seqs.qza \
  --o-alignment aligned-rep-seqs.qza \
  --o-masked-alignment masked-aligned-rep-seqs.qza \
  --o-tree unrooted-tree.qza \
  --o-rooted-tree rooted-tree.qza

Saved FeatureData[AlignedSequence] to: aligned-rep-seqs.qza
Saved FeatureData[AlignedSequence] to: masked-aligned-rep-seqs.qza
Saved Phylogeny[Unrooted] to: unrooted-tree.qza
Saved Phylogeny[Rooted] to: rooted-tree.qza


QIIME 2’s diversity analyses are available through the q2-diversity plugin, which supports computing alpha and beta diversity metrics, applying related statistical tests, and generating interactive visualizations. We’ll first apply the core-metrics-phylogenetic method, which rarefies a FeatureTable[Frequency] to a user-specified depth, computes several alpha and beta diversity metrics, and generates principle coordinates analysis (PCoA) plots using Emperor for each of the beta diversity metrics. The metrics computed by default are:

Alpha diversity

Shannon’s diversity index (a quantitative measure of community richness)

Observed Features (a qualitative measure of community richness)

Faith’s Phylogenetic Diversity (a qualitative measure of community richness that incorporates phylogenetic relationships between the features)

Evenness (or Pielou’s Evenness; a measure of community evenness)

Beta diversity

Jaccard distance (a qualitative measure of community dissimilarity)

Bray-Curtis distance (a quantitative measure of community dissimilarity)

unweighted UniFrac distance (a qualitative measure of community dissimilarity that incorporates phylogenetic relationships between the features)

weighted UniFrac distance (a quantitative measure of community dissimilarity that incorporates phylogenetic relationships between the features)

An important parameter that needs to be provided to this script is --p-sampling-depth, which is the even sampling (i.e. rarefaction) depth. Because most diversity metrics are sensitive to different sampling depths across different samples, this script will randomly subsample the counts from each sample to the value provided for this parameter.

Choosing this value is tricky. We recommend making your choice by reviewing the information presented in the table.qzv file that was created above. Choose a value that is as high as possible (so you retain more sequences per sample) while excluding as few samples as possible.



In [ ]:
!qiime diversity core-metrics-phylogenetic \
  --i-phylogeny rooted-tree.qza \
  --i-table table.qza \
  --p-sampling-depth 1103  \
  --m-metadata-file sample-metadata.tsv \
  --output-dir core-metrics-results

Saved FeatureTable[Frequency] to: core-metrics-results/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: core-metrics-results/evenness_vector.qza
Saved DistanceMatrix to: core-metrics-results/unweighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results/weighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results/jaccard_distance_matrix.qza
Saved DistanceMatrix to: core-metrics-results/bray_curtis_distance_matrix.qza
Saved PCoAResults to: core-metrics-results/unweighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results/weighted_unifrac_pcoa_results.qza
Saved PCoAResults to: core-metrics-results/jaccard_pcoa_results.qza
Saved PCoAResults to: core-metrics-results/bray_curtis_pcoa_re

### Alpha rarefaction plotting

In [ ]:
!qiime diversity alpha-rarefaction \
  --i-table table.qza \
  --i-phylogeny rooted-tree.qza \
  --p-max-depth 4000 \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization alpha-rarefaction.qzv

Saved Visualization to: alpha-rarefaction.qzv


The visualization will have two plots. The top plot is an alpha rarefaction plot, and is primarily used to determine if the richness of the samples has been fully observed or sequenced. If the lines in the plot appear to “level out” (i.e., approach a slope of zero) at some sampling depth along the x-axis, that suggests that collecting additional sequences beyond that sampling depth would not be likely to result in the observation of additional features. If the lines in a plot don’t level out, this may be because the richness of the samples hasn’t been fully observed yet (because too few sequences were collected), or it could be an indicator that a lot of sequencing error remains in the data (which is being mistaken for novel diversity).

The bottom plot in this visualization is important when grouping samples by metadata. It illustrates the number of samples that remain in each group when the feature table is rarefied to each sampling depth. If a given sampling depth d is larger than the total frequency of a sample s (i.e., the number of sequences that were obtained for sample s), it is not possible to compute the diversity metric for sample s at sampling depth d. If many of the samples in a group have lower total frequencies than d, the average diversity presented for that group at d in the top plot will be unreliable because it will have been computed on relatively few samples. When grouping samples by metadata, it is therefore essential to look at the bottom plot to ensure that the data presented in the top plot is reliable.

### Taxonomic classification

This step deals with the exploration of taxonomic composition of the samples, and again relate that to sample metadata. The first step in this process is to assign taxonomy to the sequences in our FeatureData[Sequence] QIIME 2 artifact. We’ll do that using a pre-trained Naive Bayes classifier and the q2-feature-classifier plugin. This classifier was trained on the Greengenes 13_8 99% OTUs, where the sequences have been trimmed to only include 250 bases from the region of the 16S that was sequenced in this analysis (the V4 region, bound by the 515F/806R primer pair). We’ll apply this classifier to our sequences, and we can generate a visualization of the resulting mapping from sequence to taxonomy.

In [ ]:
!wget \
  -O "gg-13-8-99-515-806-nb-classifier.qza" \
  "https://data.qiime2.org/classifiers/sklearn-1.4.2/greengenes/gg-13-8-99-515-806-nb-classifier.qza"

In [ ]:
!qiime feature-classifier classify-sklearn \
  --i-classifier gg-13-8-99-515-806-nb-classifier.qza \
  --i-reads rep-seqs.qza \
  --o-classification taxonomy.qza

!qiime metadata tabulate \
  --m-input-file taxonomy.qza \
  --o-visualization taxonomy.qzv

Saved FeatureData[Taxonomy] to: taxonomy.qza
Saved Visualization to: taxonomy.qzv


### Visualizing taxonomy bar plots

In [ ]:
!qiime taxa barplot \
  --i-table table.qza \
  --i-taxonomy taxonomy.qza \
  --m-metadata-file sample-metadata.tsv \
  --o-visualization taxa-bar-plots.qzv